In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from pathlib import Path
import sys
import shutil
import os
def ensure_tests_on_path(max_up=10):
    p = Path.cwd().resolve()
    for _ in range(max_up):
        if (p / 'tests').is_dir():
            sys.path.insert(0, str(p))
            return str(p)
        if p.parent == p:
            break
        p = p.parent
    # fallback: insert cwd
    sys.path.insert(0, str(Path.cwd().resolve()))
    return str(Path.cwd().resolve())

root_added = ensure_tests_on_path()
print("Added to sys.path:", root_added)

Added to sys.path: /Users/ashmi/Code/Scripts/phd/travel-crs


In [9]:
PAPER_FIG_LOCATION = os.getenv("PAPER_LOCATION")
# PAPER_FIG_LOCATION

In [4]:
COMPARISON_MAP = {
    "much more l1 than l2": 2,
    "much more l2 than l1": -2,
    "slightly more l1": 1,
    "slightly more l1 than l2": 1,
    "slightly more l2 than l1": -1,
    "slightly more l2": -1,
    "about the same": 0,
    "not sure / don't know": -3,
    "not sure / don’t know": -3,
    "not sure": -3,
    "don't know": -3
}


In [5]:
%load_ext autoreload
%autoreload 2

In [15]:
import json
from tests.analysis.src.quantitative import *
from tests.analysis.src.utils import load_listwise_evaluations_df, add_numeric_scores, set_paper_style

In [7]:
def show_stats(version="v1"):

    if version == "v1":
        v_text = "L1: Gemini-2.5-flash, L2: GPT-4o-mini"
    else:
        v_text = "L1: Claude-4-sonnet, L2: Qwen3-80B"
    gemini_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/calibration_gemini_{version}.json'
    gpt_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/calibration_gpt5_{version}.json'
    deepseek_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/calibration_deepseek_{version}.json'
    
    df_gpt = load_listwise_evaluations_df(
        gpt_eval_path, model_name="gpt5", version=version)
    df_gemini = load_listwise_evaluations_df(
        gemini_eval_path, model_name="gemini2.5pro", version=version)
    df_deepseek = load_listwise_evaluations_df(deepseek_eval_path, model_name="deepseek", version=version)
    
    print(f"\n=== Confidence-weighted Mean Scores GPT: {version} ===")
    print(compute_stats_scores(df_gpt, with_confidence=True))
    
    print(f"\n=== Confidence-weighted Mean Scores Gemini {version}===")
    print(compute_stats_scores(df_gemini, with_confidence=True))
    
    print(f"\n=== Confidence-weighted Mean Scores Deepseek {version}===")
    print(compute_stats_scores(df_deepseek, with_confidence=True))

    compute_kruskal_test(df_gpt, df_gemini, df_deepseek)
    print("\n\n============ xxxxxxx ============")


In [8]:
show_stats(version="v1")
# show_stats(version="v2")

ℹ️ Loaded 101 successful evaluation entries for gpt5, version v1.
⚠️ Error parsing entry 15: 'str' object has no attribute 'get'


OSError: Cannot save file into a non-existent directory: 'data/conv-trs/ecir-2026/direct-reasoner/cleaned'

In [ ]:
df_gpt.columns

## Agreement

In [ ]:
version="v1"
def agreement_analysis(version="v1"):
    gemini_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/calibration_gemini_{version}.json'
    gpt_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/calibration_gpt5_{version}.json'
    deepseek_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/calibration_deepseek_{version}.json'

    print("=========\n\n")
    
    df_gpt = load_listwise_evaluations_df(
            gpt_eval_path, model_name="gpt5", version=version)
    df_gemini = load_listwise_evaluations_df(
        gemini_eval_path, model_name="gemini2.5pro", version=version)
    df_deepseek = load_listwise_evaluations_df(deepseek_eval_path, model_name="deepseek", version=version)
    
    compute_agreement(df_gpt, df_gemini, df_deepseek)

In [ ]:
agreement_analysis("v1")

In [26]:
import matplotlib.pyplot as plt
import numpy as np

# Define categories (x-axis labels)
categories = ["All agree", "GPT=Gem", "GPT=DS", "Gem=DS", "None agree"]

# Agreement percentages for vOne (from your data)
data_v1 = {
    "Relevance":       [4.95, 35.64, 1.98, 20.79, 36.63],
    "Diversity":       [20.79, 24.75, 17.82, 10.89, 25.74],
    "Popularity":      [13.86, 23.76, 18.81, 10.89, 32.67],
    "Sustainability":  [12.87, 28.71, 12.87, 14.85, 30.69],
    "Best List":       [59.41, 17.82, 8.91, 13.86, 0.00]
}

# Example placeholders for vTwo (replace with actual values when ready)
data_v2 = {
    "Relevance":       [4.00, 46, 7.00, 19.00, 37.00],
    "Diversity":       [13.00, 41.00, 30.00, 31.00, 25.00],
    "Popularity":      [24.00, 58.00, 35.00, 28.00, 28.00],
    "Sustainability":  [19.00, 45.00, 35.00, 31.68, 27.00],
    "Best List":       [57.00, 82.00, 65.00, 67.32, 0.00]
}

In [28]:
def wrap_labels(labels, max_words_per_line=2):
    """Insert line breaks in long category labels."""
    wrapped = []
    for label in labels:
        words = label.split()
        wrapped_label = "\n".join(
            [" ".join(words[i:i+max_words_per_line]) for i in range(0, len(words), max_words_per_line)]
        )
        wrapped.append(wrapped_label)
    return wrapped

In [30]:
# Bar chart configuration
bar_width = 0.35
x = np.arange(len(categories))
set_paper_style()
# Generate and save each plot
for dim in data_v1.keys():
    fig, ax = plt.subplots(figsize=(12,8))
    
    ax.bar(x - bar_width/2, data_v1[dim], bar_width, label='before', color='tab:blue')
    ax.bar(x + bar_width/2, data_v2[dim], bar_width, label='after', color='tab:orange')
    
    # ax.set_title(f"{dim} Agreement Comparison", fontsize=12, fontweight='bold')
    # ax.set_xticks(x)
    # ax.set_xticklabels(categories, rotation=30, ha='right')
    wrapped_categories = wrap_labels(categories, max_words_per_line=2)
    ax.set_xticks(x)
    ax.set_xticklabels(wrapped_categories, rotation=30, ha='center')
    ax.set_ylabel("Percentage (%)")
    ax.set_ylim(0, 100)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(f"../../plots/pdf/agreement_{dim.lower().replace(' ', '_')}.pdf", dpi=300)
    shutil.copy(f"../../plots/pdf/agreement_{dim.lower().replace(' ', '_')}.pdf", f"{PAPER_FIG_LOCATION}/figures/agreement_{dim.lower().replace(' ', '_')}.pdf")
    plt.savefig(f"../../plots/png/agreement_{dim.lower().replace(' ', '_')}.png", dpi=300)
    plt.close(fig)  # close to avoid memory issues when looping

print("✅ Saved all plots")

✅ Saved all plots


## Pairwise Correlation

In [ ]:
version="v1"
def show_correlations(version="v1"):
    gemini_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/gemini_eval_{version}.json'
    gpt_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/gpt5_eval_{version}.json'
    deepseek_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/deepseek_eval_{version}.json'
    
    print("=========\n\n")
    
    df_gpt = load_listwise_evaluations_df(
            gpt_eval_path, model_name="gpt5", version=version)
    df_gemini = load_listwise_evaluations_df(
        gemini_eval_path, model_name="gemini2.5pro", version=version)
    df_deepseek = load_listwise_evaluations_df(deepseek_eval_path, model_name="deepseek", version=version)
    
    print("\n Models: GPT, Gemini")
    print(pairwise_correlation(df_gpt, df_gemini, "GPT", "Gemini"))
    
    print("\n Models: GPT, Deepseek")
    print(pairwise_correlation(df_gpt, df_deepseek, "GPT", "Deepseek"))
    
    print("\n Models: Gemini, Deepseek")
    print(pairwise_correlation(df_gemini, df_deepseek, "Gemini", "Deepseek"))


In [ ]:
show_correlations("v1")

In [ ]:
show_correlations("v2")

## Self Bias